# Exercise 2: Augmentation operations and robustness

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Speech Technologies · version 0.1.0*

> Apply resampling speed perturbation, pitch-preserving time stretching, pitch shifting and additive noise separately. Verify what each operation changes (duration, fundamental frequency, signal-to-noise ratio) and measure the recogniser's sensitivity to each before using any of them in training.

**Setup.** `pip install sherpa-onnx==1.13.8 librosa==0.11.0 soundfile requests numpy`. Everything runs on the CPU without a GPU or the Hugging Face Hub:

- **ASR:** OpenAI Whisper *small*, as the int8 ONNX export published in the sherpa-onnx GitHub release `asr-models` (639 MB archive), decoded greedily by sherpa-onnx.
- **TTS:** the Piper VITS voice `kk_KZ-issai-high` (six speakers, trained from scratch on the ISSAI KazakhTTS data, CC BY 4.0 according to its model card) and, in Exercise 4, `tr_TR-dfki-medium` (one speaker, CC BY-NC-SA 4.0), from the sherpa-onnx release `tts-models`.
- **Text:** the Kazakh and Turkish sides of the X-WMT English–Kazakh and English–Turkish test sets in the commit-pinned TIL release (MIT licence), with the spaces around punctuation removed.

Archives are checked against the SHA-256 digests in the code and cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), together with every synthesised waveform and every transcript, so a rerun reuses them. **The test speech is synthetic.** It comes from a TTS model trained on read studio speech; its error rates say nothing about Whisper on real Kazakh recordings, and they are used here only to compare conditions on identical input. Piper samples its output (noise scale 0.667), so a fresh cache produces slightly different audio and slightly different numbers.

Each operation is applied to the same 30 clean utterances from Exercise 1 (the first cell re-creates them from the cache). The table first checks the operations themselves, then gives Whisper's error rates on the transformed audio. This is a robustness test of a fixed model, not a training ablation: it shows which variations the model is sensitive to, which is one input to deciding what to augment.

In [ ]:
import collections, hashlib, json, os, re, statistics, tarfile, time, unicodedata
from pathlib import Path
import numpy as np, requests

TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SHERPA = "https://github.com/k2-fsa/sherpa-onnx/releases/download/"
ARCHIVES = {   # sherpa-onnx release archive -> SHA-256 of the file used for the book's run
    "asr-models/sherpa-onnx-whisper-small.tar.bz2": "486a46afbb7ba798507190ffe02fea2dd726049af212e774537efac6afb210a6",
    "tts-models/vits-piper-kk_KZ-issai-high.tar.bz2": "6c59955f7f5e3fc50b130ced59a430eb4ef62d69155af4ce836db0dde4a42682",
    "tts-models/vits-piper-tr_TR-dfki-medium.tar.bz2": "b63babaefcdd63202c7079aba25b72a2013825ad5895e2aeff03d1ae5a5b3bbf",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)
SR = 16000                                     # Whisper's sampling rate
THREADS = os.cpu_count() or 1

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(url, timeout=600, stream=True) as r:
            r.raise_for_status()
            with open(str(path) + ".part", "wb") as f:
                for block in r.iter_content(1 << 20):
                    f.write(block)
        os.replace(str(path) + ".part", path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def archive(name):
    """Download, verify and unpack a sherpa-onnx release archive; return the unpacked folder."""
    path = fetch(SHERPA + name)
    if FETCHED[SHERPA + name] != ARCHIVES[name]:
        raise ValueError(f"{name}: the release file has changed; check it before using it")
    root = path.parent / path.name.removesuffix(".tar.bz2")
    if not root.exists():
        with tarfile.open(path) as t:
            t.extractall(path.parent, filter="data")
    return root

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation)."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt_side(pair, lang):
    """One side of an X-WMT test set, detokenised."""
    text = fetch(f"{TIL}xwmt/test/{pair}/{pair}.{lang}").read_text(encoding="utf-8").splitlines()
    return [detok(s) for s in text if s.strip()]

def test_sentences(pair, lang, n):
    """The first n sentences with 5-20 words and no digits (number expansion is a separate problem)."""
    return [s for s in xwmt_side(pair, lang) if not re.search(r"\d", s) and 5 <= len(s.split()) <= 20][:n]

# ---- TTS: Piper VITS voices through sherpa-onnx; every waveform is cached on disk -------------
VOICES = {"kk": "kk_KZ-issai-high", "tr": "tr_TR-dfki-medium"}
_TTS = {}

def tts(voice, noise=None):
    if (voice, noise) not in _TTS:
        import sherpa_onnx
        root = archive(f"tts-models/vits-piper-{voice}.tar.bz2")
        vits = sherpa_onnx.OfflineTtsVitsModelConfig(model=str(root / f"{voice}.onnx"), tokens=str(root / "tokens.txt"),
                                                     data_dir=str(root / "espeak-ng-data"))
        if noise is not None:                   # noise=0 makes the voice deterministic
            vits.noise_scale = vits.noise_scale_w = noise
        _TTS[voice, noise] = sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(
            model=sherpa_onnx.OfflineTtsModelConfig(vits=vits, num_threads=THREADS)))
    return _TTS[voice, noise]

def speech(text, sid=0, lang="kk", draw=0, noise=None):
    """16 kHz float32 synthetic speech for text; `draw` indexes repeated samples of the same input."""
    key = hashlib.sha1(json.dumps([VOICES[lang], sid, draw, noise, text], ensure_ascii=False).encode()).hexdigest()
    path = CACHE / "ch11-audio" / f"{key}.npy"
    if not path.exists():
        import librosa
        a = tts(VOICES[lang], noise).generate(text, sid=sid, speed=1.0)
        w = librosa.resample(np.asarray(a.samples, dtype=np.float32), orig_sr=a.sample_rate, target_sr=SR)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.save(path, w.astype(np.float32))
    return np.load(path)

# ---- ASR: Whisper small (int8 ONNX), greedy; every transcript is cached ----------------------
_ASR = {}
ASR_CACHE = CACHE / "ch11-transcripts.jsonl"

def _asr(lang):
    if lang not in _ASR:
        import sherpa_onnx
        root = archive("asr-models/sherpa-onnx-whisper-small.tar.bz2")
        _ASR[lang] = sherpa_onnx.OfflineRecognizer.from_whisper(
            encoder=str(root / "small-encoder.int8.onnx"), decoder=str(root / "small-decoder.int8.onnx"),
            tokens=str(root / "small-tokens.txt"), language=lang, task="transcribe", num_threads=THREADS)
    return _ASR[lang]

def _transcripts():
    out = {}
    if ASR_CACHE.exists():
        for line in ASR_CACHE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[r["key"]] = r
    return out
TRANSCRIPTS = _transcripts()

def transcribe(wave, lang="kk"):
    """(text, language Whisper used, decoding seconds); lang="" lets Whisper detect the language."""
    wave = np.ascontiguousarray(wave, dtype=np.float32)
    key = hashlib.sha1(wave.tobytes()).hexdigest() + "|" + lang
    if key not in TRANSCRIPTS:
        rec = _asr(lang)
        s = rec.create_stream(); s.accept_waveform(SR, wave)
        t0 = time.time(); rec.decode_stream(s)
        r = {"key": key, "text": s.result.text.strip(), "lang": s.result.lang, "sec": time.time() - t0}
        TRANSCRIPTS[key] = r
        ASR_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with ASR_CACHE.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    r = TRANSCRIPTS[key]
    return r["text"], r["lang"], r["sec"]

# ---- Scoring: the same functions as code/remaining/speech.py -----------------------------------
def normalize(text):
    # Conservative scoring: preserve letters, diacritics and apostrophes.
    text = unicodedata.normalize("NFC", text).lower()
    return " ".join("".join(c if not unicodedata.category(c).startswith("P") or c in "'ʻʼ’" else " " for c in text).split())

def distance(a, b):
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        nxt = [i]
        for j, y in enumerate(b, 1):
            nxt.append(min(nxt[-1] + 1, row[j] + 1, row[j - 1] + (x != y)))
        row = nxt
    return row[-1]

def errors(refs, hyps):
    """Corpus-level raw and normalised WER and CER (edits / reference words or characters)."""
    if len(refs) != len(hyps) or not refs:
        raise ValueError("unaligned/empty references")
    result = {}
    for name, fn in [("raw", lambda x: x), ("normalized", normalize)]:
        pairs = [(fn(r), fn(h)) for r, h in zip(refs, hyps)]
        for unit, split in [("wer", str.split), ("cer", list)]:
            denom = sum(len(split(r)) for r, h in pairs)
            edits = sum(distance(split(r), split(h)) for r, h in pairs)
            result[f"{name}_{unit}"] = edits / denom if denom else None
        result[f"{name}_empty_references"] = sum(not r for r, h in pairs)
    return result

def pct(x):
    return f"{100 * x:5.1f}%"

def audio_digest(waves):
    h = hashlib.sha256()
    for w in waves:
        h.update(np.ascontiguousarray(w, dtype=np.float32).tobytes())
    return h.hexdigest()[:16]

print("archives verified:", ", ".join(n.rsplit("/", 1)[-1] for n in ARCHIVES if archive(n)))

In [ ]:
kk_all = xwmt_side("en-kk", "kk")
TEST = test_sentences("en-kk", "kk", 60)               # 60 sentences, spoken by the six voices in turn
waves = [speech(s, sid=i % 6) for i, s in enumerate(TEST)]
hyps, langs, secs = zip(*(transcribe(w, "kk") for w in waves))
audio = sum(len(w) for w in waves) / SR
print(f"X-WMT en-kk test: {len(kk_all)} Kazakh sentences; the first {len(TEST)} with 5-20 words and no digits are used")
print(f"synthetic audio: {audio / 60:.1f} min, {min(len(w) for w in waves) / SR:.1f}-{max(len(w) for w in waves) / SR:.1f} s per item; audio digest {audio_digest(waves)}")
print(f"decoding: {sum(secs):.0f} s on {THREADS} CPU threads, real-time factor {sum(secs) / audio:.2f}")
r = errors(TEST, list(hyps))
print(f"Whisper small, language token kk, greedy:  raw WER {pct(r['raw_wer'])}  CER {pct(r['raw_cer'])}   "
      f"normalised WER {pct(r['normalized_wer'])}  CER {pct(r['normalized_cer'])}")
short = sum(len(normalize(h)) < 0.5 * len(normalize(r)) for r, h in zip(TEST, hyps))
print(f"hypotheses shorter than half their reference (normalised characters): {short} of {len(TEST)}")
print("normalised CER by voice:")
for sid in range(6):
    idx = [i for i in range(len(TEST)) if i % 6 == sid]
    e = errors([TEST[i] for i in idx], [hyps[i] for i in idx])
    print(f"  speaker {sid}: CER {pct(e['normalized_cer'])}  WER {pct(e['normalized_wer'])}  ({len(idx)} items)")
item = sorted(range(len(TEST)), key=lambda i: distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i])))
for name, i in [("best", item[0]), ("median", item[len(item) // 2]), ("worst", item[-1])]:
    c = distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i]))
    print(f"{name} item (CER {pct(c)}):\n  REF {TEST[i]}\n  HYP {hyps[i]}")

In [ ]:
import librosa
SUB = TEST[:30]
clean = [speech(s, sid=i % 6) for i, s in enumerate(SUB)]

def speed(w, f):          # resampling: f times faster AND pitch multiplied by f (Ko et al. 2015)
    return librosa.resample(w, orig_sr=SR, target_sr=int(round(SR / f)))
def stretch(w, f):        # phase-vocoder time stretch: duration / f, pitch unchanged
    return librosa.effects.time_stretch(w, rate=f)
def pitch(w, n):          # pitch shift by n semitones, duration unchanged
    return librosa.effects.pitch_shift(w, sr=SR, n_steps=n)
def noise(w, snr, seed):  # white Gaussian noise mixed at a declared signal-to-noise ratio
    n = np.random.default_rng(seed).standard_normal(len(w)).astype(np.float32)
    return w + n * np.sqrt(np.mean(w ** 2) / (np.mean(n ** 2) * 10 ** (snr / 10)))

CONDITIONS = [("clean", lambda w, i: w),
              ("speed x0.9", lambda w, i: speed(w, 0.9)), ("speed x1.1", lambda w, i: speed(w, 1.1)),
              ("stretch x0.9", lambda w, i: stretch(w, 0.9)), ("stretch x1.1", lambda w, i: stretch(w, 1.1)),
              ("pitch -2 st", lambda w, i: pitch(w, -2)), ("pitch +2 st", lambda w, i: pitch(w, 2)),
              ("noise 20 dB", lambda w, i: noise(w, 20, i)), ("noise 10 dB", lambda w, i: noise(w, 10, i)),
              ("noise 5 dB", lambda w, i: noise(w, 5, i))]

def f0(w):
    f, voiced, _ = librosa.pyin(w, fmin=60, fmax=500, sr=SR)
    return np.nanmedian(f[voiced])

base_f0 = [f0(w) for w in clean[:6]]                    # one item per voice
print(f"{len(SUB)} items; duration and median-F0 ratios measured on the first 6 (one per voice)")
print(f"{'condition':<13} {'duration':>9} {'F0':>6} {'SNR':>7} {'WER':>7} {'CER':>7}")
for name, fn in CONDITIONS:
    waves = [fn(w, i).astype(np.float32) for i, w in enumerate(clean)]
    dur = statistics.mean(len(a) / len(b) for a, b in zip(waves[:6], clean[:6]))
    ratio = statistics.mean(f0(a) / b for a, b in zip(waves[:6], base_f0))
    snr = ("" if not name.startswith("noise") else
           f"{statistics.mean(10 * np.log10(np.mean(b ** 2) / np.mean((a - b) ** 2)) for a, b in zip(waves, clean)):.1f}")
    hyps = [transcribe(w, "kk")[0] for w in waves]
    e = errors(SUB, hyps)
    print(f"{name:<13} {dur:9.2f} {ratio:6.2f} {snr:>7} {pct(e['normalized_wer'])} {pct(e['normalized_cer'])}")

# SpecAugment acts on the features inside training; sherpa-onnx computes Whisper's features itself,
# so the masks can be built and checked here but not fed to the recogniser.
mel = librosa.power_to_db(librosa.feature.melspectrogram(y=clean[0], sr=SR, n_fft=400, hop_length=160, n_mels=80))
rng = np.random.default_rng(0)
mask = np.zeros_like(mel, dtype=bool)
for _ in range(2):                                     # LD policy: 2 frequency masks (F=27), 2 time masks (T=100)
    f = rng.integers(0, 28); f0_ = rng.integers(0, 80 - f); mask[f0_:f0_ + f, :] = True
    t = rng.integers(0, min(100, int(0.2 * mel.shape[1])) + 1); t0 = rng.integers(0, mel.shape[1] - t); mask[:, t0:t0 + t] = True
print(f"SpecAugment (LD policy, time masks capped at 20% of {mel.shape[1]} frames): {mask.mean():.1%} of the "
      f"80 x {mel.shape[1]} log-mel cells masked; duration and pitch of the waveform are untouched")

## Worked solution

Output of `experiments/ch11_exercises/run_ch11_exercises.py`:

```
30 items; duration and median-F0 ratios measured on the first 6 (one per voice)
condition      duration     F0     SNR     WER     CER
clean              1.00   1.00          93.5%  54.4%
speed x0.9         1.11   0.90          93.2%  50.9%
speed x1.1         0.91   1.10          94.2%  58.0%
stretch x0.9       1.11   1.00          97.6%  60.5%
stretch x1.1       0.91   1.00          98.8%  71.1%
pitch -2 st        1.00   0.89          97.1%  67.2%
pitch +2 st        1.00   1.13          97.3%  65.3%
noise 20 dB        1.00   1.00    20.0  93.7%  55.0%
noise 10 dB        1.00   1.00    10.0  95.7%  61.6%
noise 5 dB         1.00   1.00     5.0  98.3%  64.4%
SpecAugment (LD policy, time masks capped at 20% of 444 frames): 45.8% of the 80 x 444 log-mel cells masked; duration and pitch of the waveform are untouched
```

**The operations do what their names say.** Resampling speed perturbation by 0.9 makes the audio 1.11 times longer *and* lowers the fundamental frequency to 0.90 of the original; by 1.1 it shortens to 0.91 and raises F0 to 1.10. The phase-vocoder stretch changes only duration (F0 ratio 1.00), pitch shifting by ±2 semitones changes only F0 (expected ratio 2^±2/12 = 0.89 and 1.12, measured 0.89 and 1.13) and the noise is mixed at exactly the declared SNR. Checking this is not pedantry: the chapter's distinction between speed perturbation and time stretching is exactly the difference between the first two rows.

**Whisper's sensitivity differs by operation.** Against a clean CER of 54.4% on these 30 items, resampling speed perturbation changes little (50.9% at 0.9, 58.0% at 1.1) and noise at 20 dB SNR changes CER to 55.0%, while the phase-vocoder stretch (60.5% and 71.1%), pitch shifting (67.2% and 65.3%) and noise at 10 and 5 dB (61.6%, 64.4%) hurt. Signal-processing artefacts are one possible explanation, but these measurements do not isolate them. With 30 items and no paired uncertainty analysis, the size of a difference alone does not establish statistical significance.

**What this does and does not show.** These are error rates of a *fixed* model on transformed test audio. They suggest which variations the model has not learned to ignore, but whether augmenting the training data with them helps can only be shown by training with each operation separately at the same number of updates and seeds, on the training split only, and comparing on unchanged validation audio.

**SpecAugment** masks regions of the features, not the waveform: two frequency masks of up to 27 mel bands and two time masks (capped at 20% of the frames) cover 45.8% of this utterance's log-mel cells. sherpa-onnx computes Whisper's features internally, so the masks cannot be fed to it; SpecAugment is a training-time operation inside the model's input pipeline.

## Your turn — extension

1. Add reverberation by convolving the clean audio with a room impulse response, and check that the duration grows by the length of the response.
2. Replace white noise with recorded babble or street noise at the same SNRs. Does the ranking of conditions change?
3. Repeat the noise rows with three seeds and report the spread.